# 04. Path-distance classification

**Status: deferred supporting branch.** Preprocessing and multivariate dynamic-time-warping variants are specified, and one historical training-channel output is stored. Corrected three-condition runs and signature features were not completed. This notebook supplies a reproducible future-work design; final conclusions exclude its results.


## What to run

Needs BasicMotions under `data/raw/BasicMotions/`. Three preprocessing conditions are separate runs, reported separately:

```bash
python scripts/run_classification.py --config configs/classification_basicmotions.yaml
python scripts/run_classification.py --config configs/classification_basicmotions_training_channel.yaml
python scripts/run_classification.py --config configs/classification_basicmotions_per_series.yaml
```

Raw archive run is the intended anchor because it leaves the published archive representation unchanged. Standardised variants change the distance and must be reported as separate experimental conditions. These commands are reproduction instructions; the corrected outputs are currently absent. Runs take seconds and use NumPy and aeon, with no GPU.


## 1. Purpose

Classification tests whether a path discrepancy preserves class-relevant shape without involving a trained reconstruction model. Model, optimiser, and training loss are absent, so performance changes can be attributed to representation and distance. Official train and test splits remain fixed.


## 2. Data and preprocessing

BasicMotions contains 40 training and 40 test paths. Each path $x^{(i)}\in\mathbb R^{6\times100}$ contains three accelerometer and three gyroscope channels sampled every $0.1$ seconds for ten seconds. Archive labels are resting, running, walking and badminton (Bagnall et al., 2018).

Preprocessing is an experimental condition. Raw data use $x^{(i)}$ as distributed. Training-channel standardisation calculates

$$
\mu_c=\frac{1}{n_{\mathrm{train}}T}\sum_{i=1}^{n_{\mathrm{train}}}\sum_{r=1}^{T}x^{(i)}_{c,r},\qquad
\sigma_c^2=\frac{1}{n_{\mathrm{train}}T}\sum_{i=1}^{n_{\mathrm{train}}}\sum_{r=1}^{T}(x^{(i)}_{c,r}-\mu_c)^2.
$$

Both splits then use $\widetilde x^{(i)}_{c,r}=(x^{(i)}_{c,r}-\mu_c)/\sigma_c$. This changes multivariate distance by weighting channel $c$ by $\sigma_c^{-2}$. Per-series standardisation instead uses one mean and scale for each pair $(i,c)$, calculated over time within that series. Raw, training-channel and per-series results are reported separately. Test labels affect none of these transformations.


## 3. Fixed 1-nearest-neighbour algorithm

For test path $z$, predict label of training path

$$
i^*(z)=\operatorname*{arg\,min}_{1\leq i\leq n_{\mathrm{train}}}d(z,x^{(i)}).
$$

Euclidean distance flattens channel and time coordinates:

$$
d_{\mathrm E}(x,z)=\left(\sum_{c=1}^{6}\sum_{r=1}^{100}|x_{c,r}-z_{c,r}|^2\right)^{1/2}.
$$

Following the dependent/independent distinction for multivariate elastic distances in Shifaz et al. (2023), dimension-dependent dynamic time warping uses local multichannel cost $\delta(r,s)=\|x_{:,r}-z_{:,s}\|_2^2$ and recursion

$$
D_{r,s}=\delta(r,s)+\min\{D_{r-1,s},D_{r,s-1},D_{r-1,s-1}\}.
$$

Dimension-independent dynamic time warping aligns each channel separately and adds channel costs:

$$
d_{\mathrm{DTW-I}}(x,z)=\sum_{c=1}^{6}d_{\mathrm{DTW}}(x_c,z_c).
$$

Accuracy is fraction of correct test labels. Balanced accuracy is mean class recall.


In [1]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display

result_paths = {
    name: Path(f'../results/runs/classification_basicmotions_{name}/results.json')
    for name in ('raw', 'training_channel', 'per_series')
}
rows = []
for preprocessing, path in result_paths.items():
    if not path.exists():
        continue
    report = json.loads(path.read_text())
    for result in report['results']:
        rows.append({
            'preprocessing': preprocessing,
            'distance': result['distance'],
            **result['scores'],
            'source': str(path),
        })

historical_path = Path('../results/runs/classification_basicmotions/results.json')
if historical_path.exists():
    historical = json.loads(historical_path.read_text())
    for result in historical['results']:
        rows.append({
            'preprocessing': 'training_channel_historical',
            'distance': result['distance'],
            **result['scores'],
            'source': str(historical_path),
        })

display(pd.DataFrame(rows))
display(pd.Series({
    name: ('complete' if path.exists() else 'pending')
    for name, path in result_paths.items()
}, name='corrected output status'))


,preprocessing,distance,accuracy,balanced_accuracy,source
0,training_channel_historical,euclidean,0.575,0.575,../results/runs/classification_basicmotions/re...
1,training_channel_historical,dtw,0.900,0.900,../results/runs/classification_basicmotions/re...


raw                 pending
training_channel    pending
per_series          pending
Name: corrected output status, dtype: object

## 4. Results

Only retained output predates corrected configurations. It used training-channel standardisation and reports accuracy $0.575$ for Euclidean distance and $0.900$ for a generic multivariate DTW implementation. Adjacent cell reads both values and all predictions directly from `results/runs/classification_basicmotions/results.json`; no number in this section is taken from an unretained run.

Raw and per-series diagnostic values were previously transcribed without retained result files, so they are excluded from current evidence. Corrected configurations separate raw, training-channel and per-series preprocessing and distinguish dependent from independent multivariate DTW. All three corrected outputs remain pending; adjacent status table makes this absence explicit. Classification comparison remains unresolved and is excluded from final project claims.


## 5. Signature extension

After corrected controls are stored, next representation is a truncated signature or log signature of an explicitly augmented path. Fix augmentation, interpolation, truncation level, input scaling and feature scaling using training information. Apply the same 1-nearest-neighbour rule so representation is the only changed component.

Raw signature uses increments and omits absolute level. Initial study therefore includes a base point or initial value. Time augmentation records cadence; lead-lag augmentation is a later alternative when interaction with increments is the intended feature.


## 6. Reproducibility map

- Data: `data/raw/BasicMotions/`
- Raw configuration: `configs/classification_basicmotions.yaml`
- Training-channel configuration: `configs/classification_basicmotions_training_channel.yaml`
- Per-series configuration: `configs/classification_basicmotions_per_series.yaml`
- Classification functions: `src/pathloss/classification.py`
- Runner: `scripts/run_classification.py`
- Corrected outputs: `results/runs/classification_basicmotions_{raw,training_channel,per_series}/results.json` after execution
- Historical training-channel output: `results/runs/classification_basicmotions/results.json`
- Dataset description and fixed split: Bagnall et al. (2018), keyed as `bagnall2018uea` in `../papers/references.bib`
- Dependent and independent multivariate DTW definitions: Shifaz et al. (2023), keyed as `shifaz2023elastic` in `../papers/references.bib`
